In [137]:
# Written by Sebastian Matiz
import pandas as pd
import requests
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score, roc_auc_score

from bs4 import BeautifulSoup
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb

cols_to_drop_for_player_stats = [
    "comment",
    "player.firstname",
    "player.lastname",
    "player.id",
    "team.id",
    "team.nickname",
    "team.code",
    "team.name",
    "team.logo",
    "game.id",
    "pos"
]

cols_to_drop_for_game_stats = [
    "league",
    "season",
    "stage",
    "officials",
    "timesTied",
    "leadChanges",
    "nugget",
    "date.start",
    "date.end",
    "date.duration",
    "status.clock",
    "status.halftime",
    "status.short",
    "status.long",
    "periods.current",
    "periods.total",
    "periods.endOfPeriod",
    "arena.name",
    "arena.city",
    "arena.state",
    "arena.country",
    "teams.visitors.id",
    "teams.visitors.name",
    "teams.visitors.nickname",
    "teams.visitors.code",
    "teams.visitors.logo",
    "teams.home.id",
    "teams.home.name",
    "teams.home.nickname",
    "teams.home.code",
    "teams.home.logo",
    "scores.visitors.points",
    "scores.visitors.win",
    "scores.visitors.loss",
    "scores.visitors.series.win",
    "scores.visitors.series.loss",
    "scores.visitors.linescore",
    "scores.home.points",
    "scores.home.win",
    "scores.home.loss",
    "scores.home.series.win",
    "scores.home.series.loss",
    "scores.home.linescore"
]

In [138]:
# rapidApi headers
headers = {
    "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
}

#########################################################
# team code by id
url = "https://api-nba-v1.p.rapidapi.com/teams"
response = requests.get(url, headers=headers).json()["response"]
df = pd.DataFrame(response)
df = df.loc[(df["nbaFranchise"] == True) & (df["allStar"] == False)]

team_nickname_to_id_map = {}
for index, row in df.iterrows():
    team_nickname_to_id_map.update({row["nickname"]: row["id"]})
#########################################################

In [139]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team_id):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team_id}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    # adding win col to df
    df["win"] = ""

    df.loc[
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team_id) == df["teams.home.id"])) |
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team_id) == df["teams.visitors.id"])),
        "win"
    ] = 1

    df.loc[
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team_id) == df["teams.home.id"])) |
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team_id) == df["teams.visitors.id"])),
        "win"
    ] = 0

    df["points"] = np.where(
        df["teams.home.id"] == team_id, 
        df["scores.home.points"], 
        df["scores.visitors.points"]
    )
    df["opponent.points"] = np.where(
        df["teams.home.id"] != team_id, 
        df["scores.home.points"], 
        df["scores.visitors.points"]
    )
    df["opponent.team_id"] = np.where(
        df["teams.home.id"] != team_id, 
        df["teams.home.id"], 
        df["teams.visitors.id"]
    )

    # adding home col to df
    df["home"] = ""
    
    df.loc[(int(team_id) == df["teams.home.id"]), "home"] = 1
        
    df.loc[(int(team_id) == df["teams.visitors.id"]), "home"] = 0    
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

In [140]:
#########################################################
# drop all cols from a df ###############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [141]:
#########################################################
# get top n player stats by game by #####################
def get_top_players_per_game_df(n, team_id, game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )

    df_opponent = df.loc[df["team.id"] != team_id]
    df = df.loc[df["team.id"] == team_id]
    
    df_opponent = drop_cols(df_opponent, cols_to_drop_for_player_stats)
    df = drop_cols(df, cols_to_drop_for_player_stats)
    
    df_opponent = zero_non_numeric_values(df_opponent)
    df = zero_non_numeric_values(df)
    
    df_opponent = df_opponent.add_prefix("opponent.")

    return df_opponent.nlargest(n, "opponent.plusMinus"), df.nlargest(n, "plusMinus")
#########################################################

In [142]:
#########################################################
# flatten data frames ###################################
def flatten_df(df):
    # Flatten the DataFrame
    flattened_data = {}
    for col in df.columns:
        for row in range(df.shape[0]):
            new_col_name = f"{col}{row}"
            flattened_data[new_col_name] = df[col].iloc[row]

    # Convert to DataFrame
    return pd.DataFrame([flattened_data])
#########################################################

In [143]:
#########################################################
# per game, get top 5 players by plusMinus metric #######
def get_top_five_players_per_game(team_id, game_ids): 
    top_5_players_on_team_per_game = {}
    for game_id in game_ids:
        # transform player_stats_df
        opponent_top_five_player_stats, friendly_top_five_player_stats = get_top_players_per_game_df(5, team_id, game_id)
        opponent_top_five_player_stats = flatten_df(opponent_top_five_player_stats)
        friendly_top_five_player_stats = flatten_df(friendly_top_five_player_stats)
        top_five_players_stats = pd.concat(
            [opponent_top_five_player_stats, friendly_top_five_player_stats], 
            axis=1
        )
        ###########################
        top_5_players_on_team_per_game[game_id] = top_five_players_stats          
    return top_5_players_on_team_per_game
#########################################################

In [144]:
#########################################################
# get win prc, and last 10 win prc ######################
def get_win_prc(game_df):
    total_win_prc = []
    last_ten_win_prc = []
    last_ten_games_win_loss = []
    total_games_played = []
    last_ten_win_count = 0
    total_win_count = 0
    total_game_count = 0
    
    for index, row in game_df.iterrows():
        total_game_count += 1
        last_ten_games_win_loss.append(row['win'])
        
        if row['win']:
            total_win_count += 1
            last_ten_win_count += 1
            
        total_win_prc.append(total_win_count/total_game_count)
               
        if total_game_count >= 10:
            if last_ten_games_win_loss[0]:
                last_ten_win_count -= 1
            last_ten_games_win_loss.pop(0)
            last_ten_win_prc.append(last_ten_win_count/10)
        else:
            last_ten_win_prc.append(last_ten_win_count/total_game_count)
        
        total_games_played.append(total_game_count)            
    return last_ten_win_prc, total_win_prc, total_games_played
#########################################################

In [145]:
#########################################################
# combine player stats and games features ###############
def combine_player_stats_and_games_data(games_df, player_stats_per_game):
    feature_map = []
    feature_map_cols_header = []
    for index, row in games_df.iterrows():
        game_id = row["id"]
        game_df = pd.DataFrame(row).transpose()
        players_stats_df = player_stats_per_game.get(game_id)
        if len(feature_map_cols_header) < 1:
            feature_map_cols_header = list(game_df) + list(players_stats_df)

        game_data = np.array(game_df.iloc[0])
        player_stats_data = np.array(players_stats_df.iloc[0])
        feature_map_data_row = np.concatenate((game_data, player_stats_data))
        feature_map.append(feature_map_data_row)
    return pd.DataFrame(feature_map, columns=feature_map_cols_header)
#########################################################

In [146]:
#########################################################
def get_feature_map_and_y(season, team_id):
    games_df = get_games_by_game_ids(season, team_id) 
    
    players_stats_per_games = get_top_five_players_per_game(
        team_id, 
        games_df["id"].array
    )
    
    last_ten_win_prc, total_win_prc, total_games_played = get_win_prc(games_df)
    games_df = games_df.assign(last_ten_w_prc=last_ten_win_prc)
    games_df = games_df.assign(w_prc=total_win_prc)
    games_df = games_df.assign(games_played=total_games_played)
    
    df = combine_player_stats_and_games_data(games_df, players_stats_per_games)
    df = zero_non_numeric_values(df)
    x = drop_cols(df, ["win", "id"])
    y = df["win"]
    return x, y 
#########################################################

In [147]:
#########################################################
normalization_func_by_team_nickname = {}
def get_xgb_model_and_rf_classifier_model(team_nickname, x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    # normalize feature map
    scaler = MinMaxScaler()
    x_train_normalized = pd.DataFrame(scaler.fit_transform(x_train), columns=x.columns)
    x_test_normalized = pd.DataFrame(scaler.transform(x_test), columns=x.columns)
    normalization_func_by_team_nickname.update({team_nickname: [scaler]})

    # XGB Classifier ########################################
    dtrain = xgb.DMatrix(x_train_normalized, label=y_train)
    dtest = xgb.DMatrix(x_test_normalized, label=y_test)

    params = {
        'objective': 'binary:logistic',
        'max_depth': 4,
        'eta': 0.1,
        'eval_metric': 'logloss',
    }

    num_boost_round = 100

    bst = xgb.train(params, dtrain, num_boost_round)
    
    # Predict probabilities on the test set
    y_pred_proba = bst.predict(dtest)
    
    # Calculate ROC AUC
    auc_score = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC AUC Score: {auc_score}")
    #########################################################

    # Classifier ############################################
    # training random forest classifier 
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_classifier.fit(x_train_normalized, y_train)
    
    # Make predictions on the test set
    rf_classifier_predictions = rf_classifier.predict(x_test_normalized)

    # Evaluate the model
    rf_classifier_accuracy = accuracy_score(y_test, rf_classifier_predictions)
    print(f"Accuracy: {rf_classifier_accuracy:.2f}")
    #########################################################
    
    return bst, rf_classifier
#########################################################

In [148]:
#########################################################
def zero_non_numeric_values(df):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df
#########################################################

In [149]:
#########################################################
xgb_rf_model_and_data_map = {}
def get_data_and_xgb_rf_models(season, team_nickname):
    team_id = team_nickname_to_id_map.get(team_nickname)
    x, y = get_feature_map_and_y(season, team_id)
   
    xgb, classifier = get_xgb_model_and_rf_classifier_model(team_nickname, x, y)
    xgb_rf_model_and_data_map.update({team_nickname: [xgb, classifier, x, y]})
#########################################################

In [150]:
#########################################################
def get_player_season_stats_avgs(season, player_id):
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"
    querystring = {"id":player_id,"season":season}
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    df = drop_cols(df, cols_to_drop_for_player_stats)
    df = zero_non_numeric_values(df)

    played_games = df[df['min'] > 0]    
    return played_games.mean()
#########################################################

In [151]:
#########################################################
def get_players_season_stats_avgs(season, player_ids):
    player_stats_avgs = []
    for i in player_ids:
        player_stats_avgs.append(get_player_season_stats_avgs(season, i))
    
    df = pd.DataFrame(player_stats_avgs)
    return df.sort_values(by=["plusMinus"], ascending=False)
#########################################################

In [152]:
#########################################################
def get_player_lineups_for_tn():
    url = "https://www.rotowire.com/basketball/nba-lineups.php"
    response = requests.get(url, verify=False)
    soup = BeautifulSoup(response.text, "html.parser")
    players_by_team = {}
    button_divs = soup.find_all("button", class_="see-court-on-off")

    for div in button_divs:
        nickname = div["data-nickname"]
        players_by_team.update({ nickname: [] })
        player_ids = div["data-lineup"].split(",")[:5]
        for player_id in player_ids:
            player_divs = soup.find_all("li", class_="lineup__player")
            for player_div in player_divs:
                a_tags = player_div.find_all("a", href=lambda href: href and player_id in href)
                for a_tag in a_tags:
                    if a_tag["title"] not in players_by_team.get(nickname):
                        players_by_team[nickname].append(a_tag["title"])

    return players_by_team
#########################################################

In [153]:
#########################################################
def get_players_by_team_and_season(team_id, season):
    url = "https://api-nba-v1.p.rapidapi.com/players"
    querystring = {"team": team_id,"season":season}
    return pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [154]:
#########################################################
def get_team_latest_stats(team_id, season):
    url = "https://api-nba-v1.p.rapidapi.com/teams/statistics"
    querystring = {"id": team_id,"season": season}
    return pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [155]:
#########################################################
def make_alphanum(str):
    return "".join(char.lower() for char in str if char.isalnum())
#########################################################

In [156]:
#########################################################
def filter_list_not_contain_substrs(list, substrs):
    res = []
    for i in substrs:
        for j in list:
            if make_alphanum(i) in make_alphanum(j):
                res.append(j)
    return res
#########################################################

In [157]:
#########################################################
def get_prediction_feature_map(
    training_data_tail,
    season,
    team_nickname,
    o_team_nickname,
    home
):   
    team_id = team_nickname_to_id_map.get(team_nickname)
    o_team_id = team_nickname_to_id_map.get(o_team_nickname)
    
    # getting player attributes
    df = get_players_by_team_and_season(team_id, season)  
    o_df = get_players_by_team_and_season(o_team_id, season)

    concat_names = df["firstname"] + " " + df["lastname"]
    o_concat_names = o_df["firstname"] + " " + o_df["lastname"]

    name_list = players_name_map.get(team_nickname)    
    o_name_list = players_name_map.get(o_team_nickname)

    filtered_names = filter_list_not_contain_substrs(concat_names, name_list)
    o_filtered_names = filter_list_not_contain_substrs(o_concat_names, o_name_list)
    
    df_players = df[concat_names.isin(set(filtered_names))]
    o_df_players = o_df[o_concat_names.isin(set(o_filtered_names))]
    
    team_latest_stats = get_team_latest_stats(team_id, season)
    o_team_latest_stats = get_team_latest_stats(o_team_id, season)

    team_avg_ppg = team_latest_stats.iloc[0]["points"] / team_latest_stats.iloc[0]["games"]
    o_team_avg_ppg = o_team_latest_stats.iloc[0]["points"] / o_team_latest_stats.iloc[0]["games"]
    
    player_ids = np.array(df_players["id"])
    o_player_ids = np.array(o_df_players["id"])
    
    players_stats_avgs = get_players_season_stats_avgs(season, player_ids)
    o_players_stats_avgs = get_players_season_stats_avgs(season, o_player_ids)
    o_players_stats_avgs = o_players_stats_avgs.add_prefix("opponent.")

    players_stats_avgs = flatten_df(players_stats_avgs)
    o_players_stats_avgs = flatten_df(o_players_stats_avgs)

    players_stats = pd.concat(
        [o_players_stats_avgs, players_stats_avgs], 
        axis=1
    )
    
    games_col_headers = [
        "points",
        "opponent.points", 
        "opponent.team_id",
        "home", 
        "last_ten_w_prc", 
        "w_prc", 
        "games_played"
    ]
    
    games_data = [
        team_avg_ppg, 
        o_team_avg_ppg, 
        o_team_id,
        home, 
        training_data_tail["last_ten_w_prc"].iloc[0], 
        training_data_tail["w_prc"].iloc[0],
        training_data_tail["games_played"].iloc[0] + 1
    ]
        
    feature_map_col_headers = games_col_headers + list(players_stats)
    feature_map_data_row = np.concatenate((games_data, players_stats.iloc[0]))
    return pd.DataFrame([feature_map_data_row], columns=feature_map_col_headers)    
#########################################################

In [158]:
players_name_map = get_player_lineups_for_tn()

/home/sebdb/projects/SBS_V1/lib/python3.11/site-packages/urllib3/connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.rotowire.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [159]:
# Get models for teams ##################################
season = "2023"
# print("76ers")
# get_data_and_xgb_rf_models(season, "76ers")
# print()

print("Knicks")
get_data_and_xgb_rf_models(season, "Knicks")
print()

# print("Pacers")
# get_data_and_xgb_rf_models(season, "Pacers")
# print()

# print("Thunder")
# get_data_and_xgb_rf_models(season, "Thunder")
# print()

print("Wizards")
get_data_and_xgb_rf_models(season, "Wizards")
print()

# print("Grizzlies")
# get_data_and_xgb_rf_models(season, "Grizzlies")
# print()

print("Rockets")
get_data_and_xgb_rf_models(season, "Rockets")
print()

# print("Spurs")
# get_data_and_xgb_rf_models(season, "Spurs")
# print()

# print("Celtics")
# get_data_and_xgb_rf_models(season, "Celtics")
# print()

print("Jazz")
get_data_and_xgb_rf_models(season, "Jazz")
print()

# print("Timberwolves")
# get_data_and_xgb_rf_models(season, "Timberwolves")
# print()

# print("Clippers")
# get_data_and_xgb_rf_models(season, "Clippers")
# print()

print("Bucks")
get_data_and_xgb_rf_models(season, "Bucks")
print()

print("Kings")
get_data_and_xgb_rf_models(season, "Kings")
print()

# print("Raptors")
# get_data_and_xgb_rf_models(season, "Raptors")
# print()
#########################################################

Knicks
ROC AUC Score: 0.9814814814814815
Accuracy: 1.00

Wizards
ROC AUC Score: 1.0
Accuracy: 0.93

Rockets
ROC AUC Score: 0.9400000000000001
Accuracy: 0.93

Jazz
ROC AUC Score: 1.0
Accuracy: 1.00

Bucks
ROC AUC Score: 0.9800000000000001
Accuracy: 0.93

Kings
ROC AUC Score: 1.0
Accuracy: 0.93



In [160]:
# Get models for teams ##################################
# print("Pistons")
# get_data_and_xgb_rf_models(season, "Pistons")
# print()

print("Nets")
get_data_and_xgb_rf_models(season, "Nets")
print()

print("Magic")
get_data_and_xgb_rf_models(season, "Magic")
print()

print("Nuggets")
get_data_and_xgb_rf_models(season, "Nuggets")
print()

# print("Heat")
# get_data_and_xgb_rf_models(season, "Heat")
# print()

print("Bulls")
get_data_and_xgb_rf_models(season, "Bulls")
print()

# print("Hornets")
# get_data_and_xgb_rf_models(season, "Hornets")
# print()

# print("Cavaliers")
# get_data_and_xgb_rf_models(season, "Cavaliers")
# print()

print("Pelicans")
get_data_and_xgb_rf_models(season, "Pelicans")
print()

# print("Warriors")
# get_data_and_xgb_rf_models(season, "Warriors")
# print()

print("Suns")
get_data_and_xgb_rf_models(season, "Suns")
print()

print("Mavericks")
get_data_and_xgb_rf_models(season, "Mavericks")
print()

# print("Lakers")
# get_data_and_xgb_rf_models(season, "Lakers")
# print()

print("Hawks")
get_data_and_xgb_rf_models(season, "Hawks")
print()

# print("Trail Blazers")
# get_data_and_xgb_rf_models(season, "Trail Blazers")
# print()
#########################################################

Nets
ROC AUC Score: 1.0
Accuracy: 1.00

Magic
ROC AUC Score: 1.0
Accuracy: 0.93

Nuggets
ROC AUC Score: 1.0
Accuracy: 1.00

Bulls
ROC AUC Score: 0.9285714285714286
Accuracy: 0.87

Pelicans
ROC AUC Score: 1.0
Accuracy: 1.00

Suns
ROC AUC Score: 0.9400000000000001
Accuracy: 0.87

Mavericks
ROC AUC Score: 0.9444444444444444
Accuracy: 1.00

Hawks
ROC AUC Score: 0.962962962962963
Accuracy: 0.87



In [161]:
def predict(
    season,
    nickname_0, 
    nickname_1
):
    x_0 = get_prediction_feature_map(xgb_rf_model_and_data_map.get(nickname_0)[2].tail(1), season, nickname_0, nickname_1, 0)
    x_1 = get_prediction_feature_map(xgb_rf_model_and_data_map.get(nickname_1)[2].tail(1), season, nickname_1, nickname_0, 1)
    normalization_func_0 = normalization_func_by_team_nickname.get(nickname_0)[0]
    normalization_func_1 = normalization_func_by_team_nickname.get(nickname_1)[0]
    x_0_normalized = pd.DataFrame(normalization_func_0.transform(x_0), columns=x_0.columns)
    x_1_normalized = pd.DataFrame(normalization_func_1.transform(x_1), columns=x_1.columns)
    
    print(nickname_0, " Xgb Proba: ", xgb_rf_model_and_data_map.get(nickname_0)[0].predict(x_0_normalized))
    print(nickname_1, " Xgb Proba: ", xgb_rf_model_and_data_map.get(nickname_1)[0].predict(x_1_normalized))
    print(nickname_0, " Classifier: ", xgb_rf_model_and_data_map.get(nickname_0)[1].predict(x_0_normalized))
    print(nickname_1, " Classifier: ", xgb_rf_model_and_data_map.get(nickname_1)[1].predict(x_1_normalized))
    

In [162]:
predict(season, "Kings", "Wizards")

TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'pandas.core.frame.DataFrame'>)

In [ ]:
predict(season, "Pelicans", "Magic")

In [ ]:
predict(season, "Bulls", "Rockets")

In [ ]:
predict(season, "Nets", "Bucks")

In [ ]:
predict(season, "Jazz", "Mavericks")

In [ ]:
predict(season, "Knicks", "Nuggets")

In [ ]:
predict(season, "Hawks", "Suns")